<a href="https://colab.research.google.com/github/canurdon/SnowLine/blob/main/sentinel2_snow_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='snowline-491111')

In [ ]:
!pip install earthengine-api geemap -q

In [ ]:
#create Tantalus bounding box
aoi = ee.Geometry.Rectangle([-123.15, 49.85, -122.85, 50.05])

#Verify area loaded correctly
print('area of interest defined')
print('Bounds:', aoi.bounds().getInfo())

In [ ]:
# Load sentinel-2 surface reflectance collection
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
.filterBounds(aoi) \
.filterDate('2024-10-01', '2025-03-01') \
.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
.sort('system:time_start', False)

# Get most recent image
image = s2.first()

# Print some metadata
info = image.getInfo()
print('Image ID:', info['id'])
print('Date:', ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print('Cloud cover:', image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(), '%')

In [ ]:
# Select the bands we need
# Band 3 = Green, Band 11 =SWIR
green = image.select('B3')
swir = image.select('B11')

# Compute NDSI
# Formula: (Green - SWIR) / (Green + SWIR)
ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')

#Threshhold at 0.4 - standard snow detection cutoff
snow_mask = ndsi.gt(0.4)

# Visualise on an interactive map
map1 = geemap.Map()
map1.centerObject(aoi, zoom=11)

# Add layers
map1.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map1.addLayer(
    ndsi,
    {'min': -1, 'max': 1, 'palette': ['black', 'white']},
    'NDSI'
)
map1.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask'
)

map1


In [ ]:
# SCL identifies the type of terrain represented by each pixel
# Numbers we want to mask out:
# 3 – cloud shadow; 8 – cloud medium probability; 9 – cloud high probability; 10 – thin cirrus

# create variable for SCL
scl = image.select('SCL')

# create cloud mask: keep = 1; mask out = 0
cloud_mask = scl.neq(3) \
.And(scl.neq(8)) \
.And(scl.neq(9)) \
.And(scl.neq(10))

# Apply cloud mask to NDSI
ndsi_masked = ndsi.updateMask(cloud_mask)
snow_mask_masked = ndsi_masked.gt(0.4)

# Add to map
map2 = geemap.Map()
map2.centerObject(aoi, zoom=11)

map2.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map2.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask unmasked'
)
map2.addLayer(
    snow_mask_masked,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask masked'
)
map2.addLayer(
    cloud_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ffffff']},
    'Cloud mask'
)

map2

In [ ]:
import datetime

# Create 60 day window over which to assess each pixel
end_date = ee.Date(datetime.datetime.now())
start_date = end_date.advance(-60, 'day')

# Gather collection
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
.filterBounds(aoi) \
.filterDate(start_date, end_date)

# May need to add fiter for compute efficiency see DECISIONS.md

# Create unction to mask clouds and compute NDSI for each image
def process_image(image):
    scl = image.select('SCL')

    # Cloud mask
    cloud_mask = scl.neq(3) \
        .And(scl.neq(8)) \
        .And(scl.neq(9)) \
        .And(scl.neq(10))

    # Water mask
    water_mask = scl.neq(6)

    # Combined mask
    valid_mask = cloud_mask \
    .And(water_mask)

    # NDSI
    ndsi = image.normalizedDifference(['B3', 'B11']) \
               .rename('NDSI')

    # Snow mask
    snow = ndsi.gt(0.4).rename('snow')

    # Apply combined mask
    snow_masked = snow.updateMask(valid_mask)

    # Timestamp
    timestamp = image.metadata('system:time_start') \
                    .rename('timestamp') \
                    .clip(aoi)

    return snow_masked \
        .addBands(timestamp) \
        .clip(aoi) \
        .copyProperties(image, ['system:time_start'])

# Apply function to collection
processed = collection.map(process_image)

# Composite: most recent valid pixel
composite = processed.qualityMosaic('timestamp').clip(aoi)


print('composite band created')
print(' names:', composite.bandNames().getInfo())

In [ ]:
# Visualise the composite
map3 = geemap.Map()
map3.centerObject(aoi, zoom=11)

# Extract the snow band
snow_composite = composite.select('snow')

# Extract timestamp and convert to days ago
now_ms = ee.Date(datetime.datetime.now()).millis()
days_ago = composite.select('timestamp') \
.subtract(now_ms) \
.abs() \
.divide(86400000) \
.rename('days_ago')

map3.addLayer(
    snow_composite,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow composite'
)

map3.addLayer(
    days_ago,
    {'min': 0, 'max': 60, 'palette': ['00ff00', 'ffff00', 'ff0000']},
    'Pixel age (days)'
)

map3

In [ ]:
print('Composite geometry:', composite.geometry().getInfo()['type'])
print('Snow band extent:', composite.select('snow').geometry().getInfo()['type'])

In [ ]:
print(composite.geometry().getInfo()['coordinates'])

In [ ]:
# Sample some pixel values from days_ago
sample = days_ago.sample(
    region=aoi,
    scale=100,
    numPixels=20,
    geometries=True
)

values = sample.getInfo()
for f in values['features']:
    print('days ago:', f['properties']['days_ago'])

In [ ]:
print('Images in collection:', collection.size().getInfo())

# Print the date of each image
dates = collection.aggregate_array('system:time_start') \
    .map(lambda t: ee.Date(t).format('YYYY-MM-dd'))
print('Available passes:', dates.getInfo())